# Prompt

In [1]:
# dependencies
%pip install datasets -q transformers accelerate torch

In [2]:
# Preview Datasets
from datasets import load_dataset

# Generate JSON Output
import json

# Load model
import torch
import re
from transformers import pipeline

In [3]:
# Dataset configurations
DATASETS = {
    "gsm8k": ("gsm8k", "main"),
    "asdiv": ("yimingzhang/asdiv", None),
    "metamathqa": ("meta-math/MetaMathQA", None),
    "openmathinstruct": ("nvidia/OpenMathInstruct-1", None), # Can change to nvidia/OpenMathInstruct-2 for larger datasets
}

DATASET_CONFIG = {
    "gsm8k": {
        "path": "gsm8k",
        "subset": "main",
        "split": "test",
        "extract_question": lambda x: x["question"],
        "extract_answer": lambda x: x["answer"]
    },
    "asdiv": {
        "path": "yimingzhang/asdiv",
        "subset": None,
        "split": "test",
        "extract_question": lambda x: x["text"].replace("Question:", "").replace("Answer:", "").strip(),
        "extract_answer": lambda x: x["label"]   # clean numeric answer
    },
    "metamathqa": {
        "path": "meta-math/MetaMathQA",
        "subset": None,
        "split": "train", # No testing dataset available
        "extract_question": lambda x: x["original_question"],
        "extract_answer": lambda x: x["response"]
    },
    "openmathinstruct": {
        "path": "nvidia/OpenMathInstruct-1", # Can change to nvidia/OpenMathInstruct-2 for larger datasets
        "subset": None,
        "split": "validation",
        "extract_question": lambda x: x["question"],
        "extract_answer": lambda x: x["expected_answer"]
    }
}

MODEL_NAME = "Qwen/Qwen2-1.5B"

In [4]:
# Key Functions

# PROMPT
# Build Prompt
def build_prompt(question, name):
    return f"""
You are a deterministic mathematics reasoning engine.

Solve the following math problem step by step using clear logical reasoning.
Show all intermediate reasoning clearly.
At the end, provide the final numerical answer.

Follow these steps:
1. Analyze the problem.
2. Compute step by step.
3. Double-check calculations.
4. Provide final result.

Important rules:
- Do NOT include any text outside the JSON.
- Do NOT use markdown.
- Ensure the final answer is a single number or expression.
- Do NOT restate the instructions.

Output your response strictly in the following JSON format:
{{
  "dataset": "{name}",
  "question": "...",
  "reasoning": "...",
  "final_answer": "..."
}}

Problem:
{question}
"""

# MODEL
# Input prompt to model
def ask_model(prompt, max_tokens=256):
    output = pipe(
        prompt,
        max_new_tokens=max_tokens,
        do_sample=False,
        temperature=0.0,
        return_full_text=False
    )

    return output[0]["generated_text"].strip()


def ask_model_batch(prompts, max_tokens=256):
    outputs = pipe(
        prompts,
        max_new_tokens=max_tokens,
        do_sample=False,
        temperature=0.0,
        return_full_text=False
    )

    results = []
    for output in outputs:
        payload = output[0] if isinstance(output, list) else output
        results.append(payload.get("generated_text", "").strip())
    return results


def extract_number(text):
    numbers = re.findall(r"-?\d+(?:\.\d+)?", text)
    if numbers:
        return numbers[-1]
    return None


def parse_model_response(response_text):
    # Try to parse a JSON object from model output; fall back to raw text.
    try:
        json_start = response_text.find("{")
        json_end = response_text.rfind("}")
        if json_start != -1 and json_end != -1 and json_end > json_start:
            candidate = response_text[json_start:json_end + 1]
            parsed = json.loads(candidate)
            reasoning = str(parsed.get("reasoning", "")).strip()
            final_answer = str(parsed.get("final_answer", "")).strip()
            if reasoning or final_answer:
                return reasoning or response_text, final_answer or (extract_number(response_text) or "")
    except Exception:
        pass

    fallback_answer = extract_number(response_text) or ""
    return response_text, fallback_answer


def evaluate_math_question(question, correct_answer):
    response = ask_model(question)
    predicted = extract_number(response)

    print("Question:", question)
    print("Model Response:", response)
    print("Extracted Answer:", predicted)
    print("Correct Answer:", correct_answer)

    if str(predicted) == str(correct_answer):
        print("Correct\n")
        return True
    else:
        print("Incorrect\n")
        return False

In [5]:
for name, (path, subset) in DATASETS.items():
    print("\n" + "=" * 70)
    print(f"Dataset: {name}")
    print("=" * 70)

    try:
        # Load dataset
        if subset:
            dataset = load_dataset(path, subset)
        else:
            dataset = load_dataset(path)

        print("Available splits:", list(dataset.keys()))

        for split in dataset.keys():
            print(f"\n--- Split: {split} ---")
            print("Number of samples:", len(dataset[split]))

            # Column names
            print("Column names:", dataset[split].column_names)

            # Feature types (schema)
            print("Features:")
            print(dataset[split].features)

            # Print one example
            print("\nFirst sample:")
            print(dataset[split][0])

    except Exception as e:
        print("Error loading dataset:", e)


Dataset: gsm8k


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

...

correct_solutions/validation.jsonl:   0%|          | 0.00/203M [00:00<?, ?B/s]

incorrect_solutions/validation.jsonl:   0%|          | 0.00/981M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Available splits: ['train', 'validation']

--- Split: train ---
Number of samples: 7321344
Column names: ['question', 'expected_answer', 'predicted_answer', 'error_message', 'is_correct', 'generation_type', 'dataset', 'generated_solution']
Features:
{'question': Value('string'), 'expected_answer': Value('string'), 'predicted_answer': Value('string'), 'error_message': Value('string'), 'is_correct': Value('bool'), 'generation_type': Value('string'), 'dataset': Value('string'), 'generated_solution': Value('string')}

First sample:
{'question': 'Martha has 18 crayons. She lost half of them, so she bought a new set of 20 crayons. How many crayons in total does Martha have after the purchase?', 'expected_answer': '29', 'predicted_answer': '29', 'error_message': '', 'is_correct': True, 'generation_type': 'masked_reference_solution', 'dataset': 'gsm8k', 'generated_solution': "Let's solve this problem using Python code.\n<llm-code>\namount_of_lost_crayons = 18 / 2\namount_of_new_crayons = 20\nt

In [6]:
# Phase 1: Prepare all dataset questions in memory
math_outputs = []

for name, config in DATASET_CONFIG.items():
    print(f"Preparing {name}...")

    try:
        if config["subset"]:
            dataset = load_dataset(config["path"], config["subset"])
        else:
            dataset = load_dataset(config["path"])

        split_data = dataset[config["split"]]

        for sample in split_data:
            question = config["extract_question"](sample)
            answer = config.get("extract_answer", lambda x: None)(sample)

            math_outputs.append({
                "dataset": name,
                "question": question,
                "expected_answer": answer,
                "prompt": build_prompt(question, name)
            })

        print(f"Prepared {len(split_data)} records from {name}.")
    except Exception as e:
        print(f"Error preparing {name}: {e}")

print(f"Total prepared records: {len(math_outputs)}")

Preparing gsm8k...
Prepared 1319 records from gsm8k.
Preparing asdiv...
Prepared 301 records from asdiv.
Preparing metamathqa...
Prepared 395000 records from metamathqa.
Preparing openmathinstruct...
Prepared 1127629 records from openmathinstruct.
Total prepared records: 1524249


# Model

In [7]:
device = 0 if torch.cuda.is_available() else -1

pipe = pipeline(
    "text-generation",
    model=MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=device
)

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
# Phase 2: Run model over all prepared questions and write final JSON
BATCH_SIZE = 4
MAX_NEW_TOKENS = 256

model_outputs = []
total_records = len(math_outputs)

for start in range(0, total_records, BATCH_SIZE):
    end = min(start + BATCH_SIZE, total_records)
    batch = math_outputs[start:end]
    batch_prompts = [item["prompt"] for item in batch]

    try:
        batch_responses = ask_model_batch(batch_prompts, max_tokens=MAX_NEW_TOKENS)
    except Exception as e:
        print(f"Batch failed for records {start} to {end - 1}: {e}")
        batch_responses = [""] * len(batch)

    for item, response_text in zip(batch, batch_responses):
        reasoning, final_answer = parse_model_response(response_text)
        model_outputs.append({
            "dataset": item["dataset"],
            "question": item["question"],
            "reasoning": reasoning,
            "final_answer": final_answer,
            "expected_answer": item["expected_answer"]
        })

    print(f"Processed {end}/{total_records} records")

with open("math_outputs.json", "w", encoding="utf-8") as f:
    json.dump(model_outputs, f, indent=4, ensure_ascii=False)

print(f"Saved {len(model_outputs)} records to math_outputs.json")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KeyboardInterrupt: 

# Past Test Cases

In [ ]:
# Test Case 1
question = "If I have 5 red apples and 7 green apples, how many apples do I have?"

response = ask_model(question)

print("Model Output:")
print(response)

In [ ]:
evaluate_math_question(
    "If I have 5 red apples and 7 green apples, how many apples do I have?",
    12
)

In [ ]:
questions = [
    ("If I have 5 red apples and 7 green apples, how many apples do I have?", 12),
    ("John has 10 books and buys 3 more. How many books does he have?", 13),
    ("There are 20 birds, 5 fly away. How many remain?", 15),
    ("Sarah has 8 candies and eats 2. How many are left?", 6),
]

correct = 0

for q, ans in questions:
    if evaluate_math_question(q, ans):
        correct += 1

accuracy = (correct / len(questions))*100
print(f"Final Accuracy: {accuracy:.2f}")